# Factor-Split QQA, guarantees, Study/Trial, and Benchmark Hub

This credential-free notebook keeps QQA as the primal population engine while making factor backends, exact certification, black-box trials, and benchmark claims explicit.

In [ ]:
import networkx as nx
import torch

import qqa

qqa.fix_seed(7)
print("QQA4CO", qqa.__version__)

## 1. Inspect the QQA-centred execution DAG

In [ ]:
problem = qqa.MaxCut(nx.cycle_graph(12))
plan = qqa.plan(problem, profile="balanced", exact_backend="none", replicas=16, epochs=20)
print(plan.explain())
assert plan.stage("qqa-primal").role == "population-primal-search"

In [ ]:
result = qqa.solve(
    problem, profile="fast", exact_backend="none",
    replicas=16, epochs=20, seed=7, return_population=True,
)
print({
    "status": result.status.value,
    "guarantee": result.guarantee_level.value,
    "feasibility": result.violations.status.value,
    "objective": result.objective_value,
    "internal_energy": result.internal_energy,
    "merit": result.merit_value,
})

## 2. Compile factor backends and verify value/gradient parity

In [ ]:
from qqa.model import LinearFactor, ModelIR, ObjectiveIR, VariableBlock, compile_execution_plan

model = ModelIR(
    (VariableBlock("x", "binary", (3,)),),
    ObjectiveIR((LinearFactor(torch.arange(3), torch.tensor([2.0, -1.0, 0.5])),)),
)
execution = compile_execution_plan(model, device="cpu")
value, gradient = execution.internal_value_and_grad(torch.tensor([[0.2, 0.8, 0.4]]))
print(execution.to_dict())
print("value", value.tolist(), "gradient", gradient.tolist())

## 3. Run a resumable black-box Study with QQA batch acquisition

In [ ]:
blackbox = qqa.BlackBoxProblem(
    [qqa.Real("ratio", 0.0, 1.0), qqa.Integer("units", 0, 5)],
    lambda point: (point["ratio"] - 0.3) ** 2 + (point["units"] - 2) ** 2,
    name="portable-process-design",
)
study = qqa.create_study(blackbox, seed=7)
study_result = study.optimize(
    budget=12, initial_points=6, batch_size=3, candidate_pool=64,
    qqa_acquisition_epochs=5, qqa_acquisition_replicas=8,
)
print(study.best_trial.point, study.best_trial.value)
print(study_result.metadata["acquisition_optimizer"])

## 4. Report a paired benchmark interval

Benchmark Hub manifests contain no machine paths, hostnames, credentials, or private endpoints.

In [ ]:
from qqa.benchmarking import paired_metric_summary

comparison = paired_metric_summary(
    candidate=(0.31, 0.25, 0.28, 0.22, 0.27),
    baseline=(0.38, 0.29, 0.30, 0.26, 0.31),
    bootstrap_samples=500, seed=7,
)
print(comparison)